### <font color='#0400ffff'>**Index** </font><a class='anchor' id='toc'></a>

- [1. Introduction](#1)
- [2. Libraries & Setup](#2)
- [3. Data Ingestion with RDDs](#3)
  - [3.1 The Multi-Line CSV Challenge](#3_1)
  - [3.2 Loading and Parsing the Raw File](#3_2)
  - [3.3 Inspecting Raw Parsed Data](#3_3)
- [4. RDD Transformations — Cleaning](#4)
  - [4.1 Per-Field Parsing Helpers](#4_1)
  - [4.2 Applying `map()` and `filter()`](#4_2)
- [5. RDD Analytics](#5)
  - [5.1 Players per Nationality — `reduceByKey`](#5_1)
  - [5.2 Position Frequency — `flatMap` + `reduceByKey`](#5_2)
  - [5.3 Filter Examples](#5_3)
- [6. RDD → DataFrame](#6)
- [7. Feature Engineering (DataFrame stage)](#7)
- [8. Persist as Parquet](#8)

<a class="anchor" id="1">

# **1. Introduction**

[Back to TOC](#toc)
</a>

This notebook covers the **Data Ingestion & Preparation** stage of the FIFA 21 pipeline.
It follows the *bronze → silver* Lakehouse pattern:

| Stage | What happens |
|-------|-------------|
| Bronze | Raw CSV loaded and parsed with RDDs |
| Silver | Cleaned, typed, feature-engineered DataFrame saved as Parquet |

**Why RDDs here?**  The raw CSV has embedded newlines inside quoted Club name fields,
making it a genuinely unstructured text-parsing problem — exactly the use case RDDs
were designed for. We use `sc.wholeTextFiles()` + Python's `csv` module to correctly
parse multi-line records, then apply `map()` for per-field cleaning, `filter()` to drop
malformed rows, and `reduceByKey` / `flatMap` for exploratory aggregations before
promoting the result to a typed DataFrame and writing Parquet.

<a class="anchor" id="2">

# **2. Libraries & Setup**

[Back to TOC](#toc)
</a>

In [ ]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0" -q

In [ ]:
# Install Java 17 (required for Spark)
!sudo apt-get update -q
!sudo apt-get install -y openjdk-17-jdk-headless -q
!java -version

In [ ]:
import os
import re
import csv
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, split, trim, regexp_extract, to_date, year, size
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("FIFA21 — RDD Ingestion & Preparation")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
sc = spark.sparkContext
print("SparkSession ready. Version:", spark.version)

<a class="anchor" id="3">

# **3. Data Ingestion with RDDs**

[Back to TOC](#toc)
</a>

<a class="anchor" id="3_1">

## **3.1 The Multi-Line CSV Challenge**

[Back to TOC](#toc)
</a>

The raw CSV has Club names wrapped in quotes with literal `\n` characters inside:
```
158023,...,"\n\n\n\nFC Barcelona",2004 ~ 2021,...
```
If we use `sc.textFile()`, Spark splits on **every** `\n` — including those inside
quoted fields — so each embedded newline becomes a broken partial row. The plain
line-by-line approach cannot recover from this without knowing which lines belong
to the same record.

**Solution:** `sc.wholeTextFiles()` loads the entire file content as a single string
per file. We then hand that string to Python's `csv.reader`, which correctly handles
RFC-4180 quoting (including multi-line fields). The result is an RDD of properly
parsed rows — one tuple per player.

This is a common real-world Spark pattern for semi-structured files where you need
a proper parser rather than just a line splitter.

<a class="anchor" id="3_2">

## **3.2 Loading and Parsing the Raw File**

[Back to TOC](#toc)
</a>

In [ ]:
# wholeTextFiles returns an RDD of (path, content) pairs — one pair per file
raw_file_rdd = sc.wholeTextFiles('./fifa21 raw data v2.csv')

# Extract the header row so we can reference column names later
file_content = raw_file_rdd.first()[1]
header = next(csv.reader(io.StringIO(file_content)))
print(f"Columns ({len(header)}):")
for i, name in enumerate(header):
    print(f"  [{i:2d}] {name}")

In [ ]:
def parse_csv_content(file_tuple):
    """flatMap function: returns all data rows from a (path, content) tuple."""
    _, content = file_tuple
    reader = csv.reader(io.StringIO(content))
    rows = list(reader)
    return rows[1:]  # skip header

# flatMap applies parse_csv_content to each file and flattens the resulting lists
# repartition distributes the parsed rows across 4 tasks for parallel processing
raw_rows_rdd = raw_file_rdd.flatMap(parse_csv_content).repartition(4)
raw_count = raw_rows_rdd.count()
print(f"Raw parsed rows: {raw_count:,}")

<a class="anchor" id="3_3">

## **3.3 Inspecting Raw Parsed Data**

[Back to TOC](#toc)
</a>

In [ ]:
# Peek at a few rows to see the raw field values before cleaning
sample = raw_rows_rdd.take(3)
for row in sample:
    print("--- Row ---")
    for name, val in zip(header, row):
        print(f"  {name}: {repr(val)}")
    print()

<a class="anchor" id="4">

# **4. RDD Transformations — Cleaning**

[Back to TOC](#toc)
</a>

From the raw inspection we can identify six categories of dirty fields:

| Field(s) | Raw example | Target |
|----------|-------------|--------|
| `Club` | `"\\n\\n\\nFC Barcelona"` | `FC Barcelona` (strip newlines) |
| `Height` | `170cm` or `5'11"` | integer cm |
| `Weight` | `72kg` or `170lbs` | integer kg |
| `Value`, `Wage`, `Release Clause`, `Hits` | `€103.5M`, `€560K`, `€0` | integer |
| `W/F`, `SM`, `IR` | `4 ★`, `5★` | integer |
| All stat columns | numeric strings | integer |

Each of these is a pure per-row transformation — ideal for `map()`.

<a class="anchor" id="4_1">

## **4.1 Per-Field Parsing Helpers**

[Back to TOC](#toc)
</a>

In [ ]:
def safe_int(value):
    if value is None or str(value).strip() == '':
        return None
    try:
        return int(str(value).strip())
    except (ValueError, TypeError):
        return None


def parse_height(value):
    """'170cm' -> 170  |  '5\'11"' -> 180 (converts feet+inches to cm)"""
    if not value or value.strip() == '':
        return None
    value = value.strip()
    if "'" in value:
        parts = re.split(r"'|\"", value)
        feet   = float(parts[0]) if parts[0] else 0.0
        inches = float(parts[1]) if len(parts) > 1 and parts[1] else 0.0
        return int(feet * 30.48 + inches * 2.54)
    return int(value.replace('cm', ''))


def parse_weight(value):
    """'72kg' -> 72  |  '170lbs' -> 77 (converts lbs to kg)"""
    if not value or value.strip() == '':
        return None
    value = value.strip()
    if 'lbs' in value:
        return int(float(value.replace('lbs', '')) * 0.453592)
    return int(value.replace('kg', ''))


def parse_money(value):
    """'\u20ac103.5M' -> 103_500_000  |  '\u20ac560K' -> 560_000  |  '\u20ac0' -> 0"""
    if not value or value.strip() == '':
        return None
    value = value.strip().replace('\u20ac', '').replace(',', '')
    try:
        if 'M' in value:
            return int(float(value.replace('M', '')) * 1_000_000)
        if 'K' in value:
            return int(float(value.replace('K', '')) * 1_000)
        return int(float(value))
    except (ValueError, TypeError):
        return None


def strip_stars(value):
    """'4 \u2605' -> 4  |  '5\u2605' -> 5  (removes star rating symbols)"""
    if not value or value.strip() == '':
        return None
    try:
        return int(re.sub(r'[^\d]', '', value))
    except (ValueError, TypeError):
        return None


# Quick sanity checks
assert parse_height("5'11\"") == 180
assert parse_height('170cm')   == 170
assert parse_weight('72kg')    == 72
assert parse_weight('170lbs')  == 77
assert parse_money('\u20ac103.5M') == 103_500_000
assert parse_money('\u20ac560K')   == 560_000
assert strip_stars('4 \u2605')     == 4
assert strip_stars('5\u2605')      == 5
print('All helper assertions passed.')

<a class="anchor" id="4_2">

## **4.2 Applying `map()` and `filter()`**

[Back to TOC](#toc)
</a>

In [ ]:
def clean_row(fields):
    """
    Transform one parsed CSV row (list of strings) into a cleaned tuple.
    Returns None for rows with fewer than 77 fields so they can be dropped.
    Column indices follow the original header order.
    """
    if len(fields) < 77:
        return None
    try:
        return (
            fields[0],                                        # ID
            fields[2],                                        # Name (LongName)
            fields[5],                                        # Nationality
            safe_int(fields[6]),                              # Age
            safe_int(fields[7]),                              # OVA
            safe_int(fields[8]),                              # POT
            re.sub(r'[\n\r]+', '', fields[9]).strip(),      # Club  <- newlines stripped
            fields[10],                                       # Contract (raw, parsed later)
            fields[11],                                       # Positions
            parse_height(fields[12]),                         # Height -> cm
            parse_weight(fields[13]),                         # Weight -> kg
            fields[14].strip(),                               # Preferred_Foot
            safe_int(fields[15]),                             # BOV
            fields[16].strip(),                               # Best_Position
            fields[17].strip() or None,                       # Joined
            fields[18].strip() or None,                       # Loan_Date_End
            parse_money(fields[19]),                          # Value
            parse_money(fields[20]),                          # Wage
            parse_money(fields[21]),                          # Release_Clause
            safe_int(fields[22]),   # Attacking
            safe_int(fields[23]),   # Crossing
            safe_int(fields[24]),   # Finishing
            safe_int(fields[25]),   # Heading_Accuracy
            safe_int(fields[26]),   # Short_Passing
            safe_int(fields[27]),   # Volleys
            safe_int(fields[28]),   # Skill
            safe_int(fields[29]),   # Dribbling
            safe_int(fields[30]),   # Curve
            safe_int(fields[31]),   # FK_Accuracy
            safe_int(fields[32]),   # Long_Passing
            safe_int(fields[33]),   # Ball_Control
            safe_int(fields[34]),   # Movement
            safe_int(fields[35]),   # Acceleration
            safe_int(fields[36]),   # Sprint_Speed
            safe_int(fields[37]),   # Agility
            safe_int(fields[38]),   # Reactions
            safe_int(fields[39]),   # Balance
            safe_int(fields[40]),   # Power
            safe_int(fields[41]),   # Shot_Power
            safe_int(fields[42]),   # Jumping
            safe_int(fields[43]),   # Stamina
            safe_int(fields[44]),   # Strength
            safe_int(fields[45]),   # Long_Shots
            safe_int(fields[46]),   # Mentality
            safe_int(fields[47]),   # Aggression
            safe_int(fields[48]),   # Interceptions
            safe_int(fields[49]),   # Positioning
            safe_int(fields[50]),   # Vision
            safe_int(fields[51]),   # Penalties
            safe_int(fields[52]),   # Composure
            safe_int(fields[53]),   # Defending
            safe_int(fields[54]),   # Marking
            safe_int(fields[55]),   # Standing_Tackle
            safe_int(fields[56]),   # Sliding_Tackle
            safe_int(fields[57]),   # Goalkeeping
            safe_int(fields[58]),   # GK_Diving
            safe_int(fields[59]),   # GK_Handling
            safe_int(fields[60]),   # GK_Kicking
            safe_int(fields[61]),   # GK_Positioning
            safe_int(fields[62]),   # GK_Reflexes
            safe_int(fields[63]),   # Total_Stats
            safe_int(fields[64]),   # Base_Stats
            strip_stars(fields[65]),  # WF  (W/F stars stripped)
            strip_stars(fields[66]),  # SM  (stars stripped)
            fields[67].strip() or None,  # AW  (A/W)
            fields[68].strip() or None,  # DW  (D/W)
            strip_stars(fields[69]),  # IR  (stars stripped)
            safe_int(fields[70]),   # PAC
            safe_int(fields[71]),   # SHO
            safe_int(fields[72]),   # PAS
            safe_int(fields[73]),   # DRI
            safe_int(fields[74]),   # DEF
            safe_int(fields[75]),   # PHY
            parse_money(fields[76]),  # Hits
        )
    except Exception:
        return None  # drop rows that fail parsing entirely

In [ ]:
# map()    applies clean_row to every element of the RDD
# filter() drops elements where clean_row returned None (malformed rows)
cleaned_rdd = (
    raw_rows_rdd
    .map(clean_row)
    .filter(lambda r: r is not None)
)

clean_count = cleaned_rdd.count()
dropped     = raw_count - clean_count
print(f"Clean rows  : {clean_count:,}")
print(f"Dropped rows: {dropped}  (malformed / too few fields)")

In [ ]:
# Spot-check the Club fix and unit conversions on a few rows
sample_cleaned = cleaned_rdd.take(3)
for row in sample_cleaned:
    # index: 1=Name, 6=Club, 9=Height, 10=Weight, 16=Value
    print(
        f"Name: {row[1]:<35}"
        f"Club: {repr(row[6]):<30}"
        f"Height: {row[9]} cm  Weight: {row[10]} kg  Value: {row[16]:,}"
    )

<a class="anchor" id="5">

# **5. RDD Analytics**

[Back to TOC](#toc)
</a>

Before promoting the data to a DataFrame we run two classic RDD aggregation patterns
to demonstrate their natural expressiveness on this dataset.

| Pattern | Use case |
|---------|----------|
| `reduceByKey` | Count players per nationality (word-count pattern) |
| `flatMap` + `reduceByKey` | Explode multi-value Positions and count each one |

<a class="anchor" id="5_1">

## **5.1 Players per Nationality — `reduceByKey`**

[Back to TOC](#toc)
</a>

`reduceByKey` is the Spark RDD equivalent of `GROUP BY col, COUNT(*)`.  For each row
we emit a `(nationality, 1)` pair, then merge the values by summing counts per key.
This is the canonical *word-count* pattern applied to a real analytical question.

In [ ]:
# (nationality, 1)  ->  reduceByKey(+)  ->  (nationality, total_count)
# r[2] = Nationality
nationality_counts = (
    cleaned_rdd
    .map(lambda r: (r[2], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

top20_nat = nationality_counts.take(20)
print(f"{'Nationality':<30} {'Players':>8}")
print('-' * 40)
for nat, cnt in top20_nat:
    print(f"{nat:<30} {cnt:>8,}")

In [ ]:
nat_df = pd.DataFrame(top20_nat, columns=['Nationality', 'Players'])

fig = px.bar(
    nat_df,
    x='Players', y='Nationality', orientation='h',
    title='Top 20 Nationalities in FIFA 21',
    color='Players', color_continuous_scale='Blues', text='Players',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

<a class="anchor" id="5_2">

## **5.2 Position Frequency — `flatMap` + `reduceByKey`**

[Back to TOC](#toc)
</a>

The `Positions` field is a comma-separated list (e.g. `"RW, ST, CF"`). We want to
count how often each individual position appears across the entire dataset.

`flatMap` lets each row emit **multiple** output records — one per position — so the
subsequent `reduceByKey` counts across all players and all their listed positions.
This is the generalised word-count pattern for multi-valued fields.

In [ ]:
# r[8] = Positions  (e.g. 'RW, ST, CF')
# flatMap: each row -> one (position, 1) pair per position listed
position_counts = (
    cleaned_rdd
    .flatMap(lambda r: [(p.strip(), 1) for p in r[8].split(',') if p.strip()])
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

all_positions = position_counts.collect()
print(f"Distinct positions found: {len(all_positions)}")
print(f"\n{'Position':<10} {'Count':>8}")
print('-' * 20)
for pos, cnt in all_positions:
    print(f"{pos:<10} {cnt:>8,}")

In [ ]:
pos_df = pd.DataFrame(all_positions, columns=['Position', 'Count'])

fig = px.bar(
    pos_df,
    x='Position', y='Count',
    title='Player Count per Position (all listed positions)',
    color='Count', color_continuous_scale='Oranges', text='Count',
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

<a class="anchor" id="5_3">

## **5.3 Filter Examples**

[Back to TOC](#toc)
</a>

`filter()` is the RDD equivalent of SQL `WHERE`. It keeps only rows where the
predicate returns `True`, making it easy to build sub-populations for further analysis.

In [ ]:
# Veterans: players aged 35 or older
# r[3] = Age
veterans = cleaned_rdd.filter(lambda r: r[3] is not None and r[3] >= 35)
print(f"Veterans (age >= 35): {veterans.count():,}")

# Age distribution within the veteran group
veteran_ages = veterans.map(lambda r: r[3]).collect()
pd.Series(veteran_ages).value_counts().sort_index().plot(
    kind='bar', figsize=(10, 4),
    title='Age distribution of players aged 35+',
    color='steelblue'
)
plt.xlabel('Age'); plt.ylabel('Players'); plt.tight_layout(); plt.show()

In [ ]:
# Elite players: OVA >= 85
# r[4] = OVA
elites = cleaned_rdd.filter(lambda r: r[4] is not None and r[4] >= 85)
print(f"Elite players (OVA >= 85): {elites.count():,}")

# Which clubs have the most elite players? (reduceByKey on the filtered sub-RDD)
elite_clubs = (
    elites
    .map(lambda r: (r[6], 1))          # r[6] = Club
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .take(10)
)
print(f"\n{'Club':<35} {'Elite players':>14}")
print('-' * 50)
for club, cnt in elite_clubs:
    print(f"{club:<35} {cnt:>14,}")

In [ ]:
# Multi-position players: listed in more than 3 positions
# r[8] = Positions
versatile = cleaned_rdd.filter(lambda r: len(r[8].split(',')) > 3)
print(f"Players listed in more than 3 positions: {versatile.count():,}")

# Show a few examples
print(f"\n{'Name':<35} Positions")
print('-' * 70)
for row in versatile.take(10):
    print(f"{row[1]:<35} {row[8]}")

<a class="anchor" id="6">

# **6. RDD → DataFrame**

[Back to TOC](#toc)
</a>

Now that the RDDs have done the text-parsing work, we promote the data to a strongly-typed
DataFrame by supplying an explicit schema. The schema enforces the types we expect and makes
the data safe for downstream SparkSQL queries.

> **Column rename note:** `W/F`, `A/W`, `D/W` become `WF`, `AW`, `DW` (the `/` slash
> causes issues in Spark SQL column references). `↓OVA` becomes `OVA`.  All other column
> names with spaces are preserved — Spark handles them fine; SQL needs backtick quoting.

In [ ]:
schema = StructType([
    StructField('ID',               StringType(),  True),
    StructField('Name',             StringType(),  True),
    StructField('Nationality',      StringType(),  True),
    StructField('Age',              IntegerType(), True),
    StructField('OVA',              IntegerType(), True),
    StructField('POT',              IntegerType(), True),
    StructField('Club',             StringType(),  True),
    StructField('Contract',         StringType(),  True),
    StructField('Positions',        StringType(),  True),
    StructField('Height',           IntegerType(), True),
    StructField('Weight',           IntegerType(), True),
    StructField('Preferred_Foot',   StringType(),  True),
    StructField('BOV',              IntegerType(), True),
    StructField('Best_Position',    StringType(),  True),
    StructField('Joined',           StringType(),  True),
    StructField('Loan_Date_End',    StringType(),  True),
    StructField('Value',            IntegerType(), True),
    StructField('Wage',             IntegerType(), True),
    StructField('Release_Clause',   IntegerType(), True),
    StructField('Attacking',        IntegerType(), True),
    StructField('Crossing',         IntegerType(), True),
    StructField('Finishing',        IntegerType(), True),
    StructField('Heading_Accuracy', IntegerType(), True),
    StructField('Short_Passing',    IntegerType(), True),
    StructField('Volleys',          IntegerType(), True),
    StructField('Skill',            IntegerType(), True),
    StructField('Dribbling',        IntegerType(), True),
    StructField('Curve',            IntegerType(), True),
    StructField('FK_Accuracy',      IntegerType(), True),
    StructField('Long_Passing',     IntegerType(), True),
    StructField('Ball_Control',     IntegerType(), True),
    StructField('Movement',         IntegerType(), True),
    StructField('Acceleration',     IntegerType(), True),
    StructField('Sprint_Speed',     IntegerType(), True),
    StructField('Agility',          IntegerType(), True),
    StructField('Reactions',        IntegerType(), True),
    StructField('Balance',          IntegerType(), True),
    StructField('Power',            IntegerType(), True),
    StructField('Shot_Power',       IntegerType(), True),
    StructField('Jumping',          IntegerType(), True),
    StructField('Stamina',          IntegerType(), True),
    StructField('Strength',         IntegerType(), True),
    StructField('Long_Shots',       IntegerType(), True),
    StructField('Mentality',        IntegerType(), True),
    StructField('Aggression',       IntegerType(), True),
    StructField('Interceptions',    IntegerType(), True),
    StructField('Positioning',      IntegerType(), True),
    StructField('Vision',           IntegerType(), True),
    StructField('Penalties',        IntegerType(), True),
    StructField('Composure',        IntegerType(), True),
    StructField('Defending',        IntegerType(), True),
    StructField('Marking',          IntegerType(), True),
    StructField('Standing_Tackle',  IntegerType(), True),
    StructField('Sliding_Tackle',   IntegerType(), True),
    StructField('Goalkeeping',      IntegerType(), True),
    StructField('GK_Diving',        IntegerType(), True),
    StructField('GK_Handling',      IntegerType(), True),
    StructField('GK_Kicking',       IntegerType(), True),
    StructField('GK_Positioning',   IntegerType(), True),
    StructField('GK_Reflexes',      IntegerType(), True),
    StructField('Total_Stats',      IntegerType(), True),
    StructField('Base_Stats',       IntegerType(), True),
    StructField('WF',               IntegerType(), True),
    StructField('SM',               IntegerType(), True),
    StructField('AW',               StringType(),  True),
    StructField('DW',               StringType(),  True),
    StructField('IR',               IntegerType(), True),
    StructField('PAC',              IntegerType(), True),
    StructField('SHO',              IntegerType(), True),
    StructField('PAS',              IntegerType(), True),
    StructField('DRI',              IntegerType(), True),
    StructField('DEF',              IntegerType(), True),
    StructField('PHY',              IntegerType(), True),
    StructField('Hits',             IntegerType(), True),
])

In [ ]:
# spark.createDataFrame wraps each tuple from the RDD into a Row using the schema
fifa = spark.createDataFrame(cleaned_rdd, schema=schema)

print(f"DataFrame shape: {fifa.count():,} rows × {len(fifa.columns)} columns")
fifa.printSchema()

In [ ]:
fifa.show(5, truncate=True)

<a class="anchor" id="7">

# **7. Feature Engineering (DataFrame stage)**

[Back to TOC](#toc)
</a>

Two features are more naturally expressed in the columnar DataFrame API than in RDD lambdas:

1. **`N_Positions`** — count how many positions each player can play.
2. **Contract features** — parse the raw `Contract` string into three clean columns:
   `Contract_Status`, `Contract_Start_Year`, `Contract_End_Year`.
   We then drop the now-redundant raw columns.

This demonstrates the natural split: **RDDs for unstructured text parsing,
DataFrames for typed column operations.**

In [ ]:
# How many positions can each player play?
# size(split(Positions, ',')) counts the comma-separated entries
fifa = fifa.withColumn('N_Positions', size(split(col('Positions'), ',')))

In [ ]:
# ── Contract feature engineering ────────────────────────────────────────

# 1. Contract_Status: 'On Loan' | 'Free' | 'Permanent'
fifa = fifa.withColumn('Contract_Status',
    when(col('Contract').contains('On Loan'), 'On Loan')
    .when(col('Contract').contains('Free'),    'Free')
    .otherwise('Permanent')
)

# 2. Contract_Start_Year: year from the 'Joined' date (e.g. 'Jul 1, 2004' -> 2004)
#    Free agents have no start year on record.
fifa = fifa.withColumn('Contract_Start_Year',
    when(col('Contract_Status') == 'Free', None)
    .otherwise(year(to_date(col('Joined'), 'MMM d, yyyy')))
)

# 3. Contract_End_Year:
#    Permanent: right side of 'YYYY ~ YYYY'
#    On Loan  : 4-digit year in the contract string
#    Free     : None
fifa = fifa.withColumn('Contract_End_Year',
    when(col('Contract_Status') == 'Permanent',
         trim(split(col('Contract'), '~')[1]).cast('int'))
    .when(col('Contract_Status') == 'On Loan',
         regexp_extract(col('Contract'), r'(\d{4})', 1).cast('int'))
    .otherwise(None)
)

# Validate the results on the top rows
fifa.select(
    'Name', 'Contract', 'Contract_Status', 'Contract_Start_Year', 'Contract_End_Year'
).show(10, truncate=False)

In [ ]:
# Drop raw columns that have been replaced by the engineered features
fifa = fifa.drop('Contract', 'Joined', 'Loan_Date_End')

print(f"Final shape: {fifa.count():,} rows × {len(fifa.columns)} columns")
fifa.printSchema()

<a class="anchor" id="8">

# **8. Persist as Parquet**

[Back to TOC](#toc)
</a>

We cache the DataFrame before writing so the row-count validation reuses the in-memory
plan rather than re-running the full ingestion pipeline.

The output is Parquet — a columnar format that:
- **Preserves the full schema** (no type re-inference needed in Notebook 2)
- **Compresses numeric columns** efficiently
- **Supports predicate pushdown** so SparkSQL analytical queries are faster

In [ ]:
fifa.cache()

output_path = './cleaned_fifa.parquet'
fifa.write.mode('overwrite').parquet(output_path)
print(f"Written to: {output_path}")

In [ ]:
# ── Validation: re-read the Parquet and confirm schema + row count ─────────
verify = spark.read.parquet(output_path)
print(f"Parquet row count : {verify.count():,}")
print(f"Parquet columns   : {len(verify.columns)}")
print()
verify.printSchema()

In [ ]:
verify.select(
    'Name', 'Club', 'Nationality', 'Age', 'OVA', 'Height', 'Weight',
    'Value', 'Wage', 'WF', 'SM', 'IR',
    'Contract_Status', 'Contract_Start_Year', 'Contract_End_Year', 'N_Positions'
).show(10, truncate=False)

**Pipeline complete.** The cleaned, typed Parquet artifact is ready for Notebook 2,
where all exploratory analysis will be carried out using SparkSQL.